# Tahap 5: Pengujian Indexing dan Traceability ID Resep
Notebook ini menguji kesiapan ChromaDB, menjalankan similarity search pada kedua collection, dan memverifikasi traceability ID resep ke PostgreSQL.

In [1]:
import os
import sys
import psycopg2
from dotenv import load_dotenv
import chromadb
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings

if sys.stdout.encoding != 'utf-8':
    try:
        sys.stdout.reconfigure(encoding='utf-8', errors='replace')
    except Exception:
        pass

torch_lib = os.path.join(sys.prefix, 'Lib', 'site-packages', 'torch', 'lib')
if os.path.exists(torch_lib) and hasattr(os, 'add_dll_directory'):
    try:
        os.add_dll_directory(torch_lib)
    except Exception:
        pass

import torch
if hasattr(torch, 'set_num_threads'):
    try:
        torch.set_num_threads(max(1, (os.cpu_count() or 4) - 1))
    except Exception:
        pass

# Inisialisasi embedding dan vectorstore
local_snapshot = os.path.expanduser(r'~/.cache/huggingface/hub/models--BAAI--bge-m3/snapshots/5617a9f61b028005a4858fdac845db406aefb181')
model_name = local_snapshot if os.path.exists(local_snapshot) else 'BAAI/bge-m3'

embedding_model = HuggingFaceEmbeddings(
    model_name=model_name,
    model_kwargs={'device': 'cpu'},
    encode_kwargs={'normalize_embeddings': True}
)

chroma_client = chromadb.PersistentClient(path="chroma_db")
resep_store = Chroma(client=chroma_client, collection_name="resep", embedding_function=embedding_model)
pedoman_store = Chroma(client=chroma_client, collection_name="pedoman_gizi", embedding_function=embedding_model)

# Inisialisasi koneksi PostgreSQL
load_dotenv(dotenv_path=".env" if os.path.exists(".env") else None)
pg_conn = psycopg2.connect(
    host=os.getenv("DB_HOST", "localhost"),
    port=os.getenv("DB_PORT", "5432"),
    dbname=os.getenv("DB_NAME", "dietmate_db"),
    user=os.getenv("DB_USER", "postgres"),
    password=os.getenv("DB_PASSWORD", "bungaalisa")
)
print("Koneksi ChromaDB dan PostgreSQL siap.")

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Koneksi ChromaDB dan PostgreSQL siap.


In [2]:
count_resep = resep_store._collection.count()
count_pedoman = pedoman_store._collection.count()

print(f"Jumlah dokumen 'resep': {count_resep} (Target: 327)")
print(f"Jumlah chunk 'pedoman_gizi': {count_pedoman} (Target: 132)")

assert count_resep == 327, f"Ekspektasi 327 resep, ditemukan {count_resep}"
assert count_pedoman == 132, f"Ekspektasi 132 chunk pedoman gizi, ditemukan {count_pedoman}"
print("Verifikasi jumlah dokumen: SUKSES (100% Cocok)")

Jumlah dokumen 'resep': 327 (Target: 327)
Jumlah chunk 'pedoman_gizi': 132 (Target: 132)
Verifikasi jumlah dokumen: SUKSES (100% Cocok)


In [3]:
query_resep = "resep makanan tinggi protein berbahan ayam"
results_resep = resep_store.similarity_search_with_score(query_resep, k=3)

print(f"Hasil pencarian untuk: '{query_resep}':")
for i, (doc, score) in enumerate(results_resep, 1):
    m = doc.metadata
    print(f"{i}. [{m['recipe_id']}] {m['nama_masakan']} ({m['kategori']})")
    print(f"   Kalori: {m['kalori_kkal']} kkal | Protein: {m['protein_g']} g | Distance: {score:.4f}")

top_doc = results_resep[0][0]
top_id = top_doc.metadata['recipe_id']
print(f"\nUji Traceability untuk ID: {top_id}")

with pg_conn.cursor() as cur:
    cur.execute("""
        SELECT recipe_id, nama_masakan, kategori, jumlah_porsi, basis_porsi,
               kalori_kkal, protein_g, karbohidrat_g, lemak_g, bahan_bahan, url
        FROM recipes
        WHERE recipe_id = %s;
    """, (top_id,))
    pg_row = cur.fetchone()

assert pg_row is not None, f"ID {top_id} tidak ditemukan di PostgreSQL!"
assert top_doc.metadata['nama_masakan'] == pg_row[1], "Nama masakan tidak cocok!"
assert abs(float(top_doc.metadata['kalori_kkal']) - float(pg_row[5])) < 0.01, "Kalori tidak cocok!"
assert abs(float(top_doc.metadata['protein_g']) - float(pg_row[6])) < 0.01, "Protein tidak cocok!"

print("Perbandingan Data:")
print(f"- recipe_id    : Chroma = {top_id} | Postgres = {pg_row[0]}")
print(f"- nama_masakan : Chroma = {top_doc.metadata['nama_masakan']} | Postgres = {pg_row[1]}")
print(f"- kategori     : Chroma = {top_doc.metadata['kategori']} | Postgres = {pg_row[2]}")
print(f"- kalori_kkal  : Chroma = {top_doc.metadata['kalori_kkal']} | Postgres = {float(pg_row[5])}")
print(f"- protein_g    : Chroma = {top_doc.metadata['protein_g']} | Postgres = {float(pg_row[6])}")
print("Uji Traceability: SUKSES (100% Cocok)")

Hasil pencarian untuk: 'resep makanan tinggi protein berbahan ayam':
1. [RCP_0073] Ayam Teriyaki (Hidangan Utama)
   Kalori: 204.0 kkal | Protein: 15.89 g | Distance: 0.7591
2. [RCP_0180] Ayam Crispy Oatmil (Lauk Pauk)
   Kalori: 285.0 kkal | Protein: 20.48 g | Distance: 0.7606
3. [RCP_0053] Ayam Rica Rica (Hidangan Utama)
   Kalori: 219.0 kkal | Protein: 17.83 g | Distance: 0.7748

Uji Traceability untuk ID: RCP_0073
Perbandingan Data:
- recipe_id    : Chroma = RCP_0073 | Postgres = RCP_0073
- nama_masakan : Chroma = Ayam Teriyaki | Postgres = Ayam Teriyaki
- kategori     : Chroma = Hidangan Utama | Postgres = Hidangan Utama
- kalori_kkal  : Chroma = 204.0 | Postgres = 204.0
- protein_g    : Chroma = 15.89 | Postgres = 15.89
Uji Traceability: SUKSES (100% Cocok)


In [4]:
query_gizi = "bagaimana rumus menghitung BMR dan TDEE?"
results_gizi = pedoman_store.similarity_search_with_score(query_gizi, k=2)

print(f"Hasil pencarian untuk: '{query_gizi}':")
for i, (doc, score) in enumerate(results_gizi, 1):
    m = doc.metadata
    print(f"\nHasil #{i} [Sumber: {m['source']}] (Distance: {score:.4f}):")
    lines = doc.page_content.strip().split('\n')
    for line in lines[:8]:
        print(f"  {line}")
    if len(lines) > 8:
        print("  ...")

pg_conn.close()
print("\nSemua pengujian berhasil diselesaikan dengan sukses.")

Hasil pencarian untuk: 'bagaimana rumus menghitung BMR dan TDEE?':

Hasil #1 [Sumber: Skripsi UII - Bab 2.1 dan Lampiran D] (Distance: 0.6225):
  // Jika BMR tidak valid, kembalikan 0 
    if (!bmr) return 0 
   
    // TDEE = BMR × faktor aktivitas 
    return Math.round(bmr * getActivityMultiplier(activityLevel)) 
  } 
  //perhitungan target kalori  
  function calculateTargetCalories(tdee: number, tujuan: DietGoal | null): 
  ...

Hasil #2 [Sumber: Skripsi UII - Bab 2.1 dan Lampiran D] (Distance: 0.6388):
  (BB = berat badan dalam kg, TB = tinggi badan dalam cm, U = usia dalam tahun). Nilai 
  BMR yang telah diperoleh merepresentasikan kebutuhan energi dasar tubuh dalam kondisi 
  istirahat. Selanjutnya, nilai tersebut digunakan untuk menghitung Total Daily Energy 
  Expenditure (TDEE), yaitu total kebutuhan energi harian yang telah memperhitungkan tingkat 
  aktivitas fisik pengguna sebagaimana ditunjukkan pada Persamaan (2.3): 
  𝑇𝐷𝐸𝐸 = 𝐵𝑀𝑅 × 𝑃𝐴𝐿 (2.3) 
  dimana PAL (Physical Acti